# Stage 1: Data Structures
This notebook generates CUSUM events and dollar bars.

In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))
from src.data_structures import cusum_filter, get_dollar_bars, calibrate_cusum_h, calibrate_dollar_bar_threshold

plt.style.use('seaborn-v0_8-darkgrid')


## 1. Load Cleaned Data


In [ ]:
df = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
df.head()


## 2. CUSUM Filter Calibration and Application


In [ ]:
target_events = 400
best_h = calibrate_cusum_h(df['Adj Close'], target_events=target_events)
print(f"Calibrated h: {best_h}")

cusum_events = cusum_filter(df['Adj Close'], best_h)
print(f"Number of CUSUM events: {len(cusum_events)}")


## 3. Dollar Bars Calibration and Generation


In [ ]:
target_bars_per_year = 252
threshold = calibrate_dollar_bar_threshold(df, target_bars_per_year=target_bars_per_year)
print(f"Calibrated Dollar Bar Threshold: {threshold}")

dollar_bars = get_dollar_bars(df, threshold)
print(f"Number of dollar bars generated: {len(dollar_bars)}")
dollar_bars.head()


## 4. Comparison: Bar Count Per Year


In [ ]:
daily_bars_per_year = len(df) / ((df.index.max() - df.index.min()).days / 365.25)
dollar_bars_per_year = len(dollar_bars) / ((dollar_bars.index.max() - dollar_bars.index.min()).days / 365.25)

print(f"Daily bars per year (avg): {daily_bars_per_year:.2f}")
print(f"Dollar bars per year (avg): {dollar_bars_per_year:.2f}")


## 5. Plots


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 18))

# a. Bar count per year comparison
years = df.index.year.unique()
daily_counts = df.groupby(df.index.year).size()
dollar_counts = dollar_bars.groupby(dollar_bars.index.year).size()

width = 0.35
axes[0].bar(years - width/2, daily_counts, width, label='Daily Bars', color='blue', alpha=0.7)
axes[0].bar(years + width/2, dollar_counts, width, label='Dollar Bars', color='orange', alpha=0.7)
axes[0].set_title('Bars Per Year: Daily vs Dollar Bars')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')
axes[0].legend()

# b. CUSUM events on price chart
axes[1].plot(df.index, df['Adj Close'], label='Adj Close', color='blue', alpha=0.5)
axes[1].scatter(cusum_events, df.loc[cusum_events, 'Adj Close'], color='red', label='CUSUM Events', zorder=5, s=10)
axes[1].set_title('CUSUM Events on Price Chart')
axes[1].set_ylabel('Price')
axes[1].set_yscale('log')
axes[1].legend()

# c. Histogram of days between CUSUM events
inter_event_days = pd.Series(cusum_events).diff().dt.days.dropna()
axes[2].hist(inter_event_days, bins=50, color='purple', alpha=0.7)
axes[2].set_title('Histogram of Days Between CUSUM Events')
axes[2].set_xlabel('Days')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## 6. Normality Test: Daily vs Dollar Bar Returns


In [ ]:
daily_returns = np.log(df['Adj Close'] / df['Adj Close'].shift(1)).dropna()
dollar_returns = np.log(dollar_bars['Close'] / dollar_bars['Close'].shift(1)).dropna()

jb_stat_daily, jb_p_daily = stats.jarque_bera(daily_returns)
jb_stat_dollar, jb_p_dollar = stats.jarque_bera(dollar_returns)

print(f"Daily Returns Jarque-Bera: stat={jb_stat_daily:.2f}, p={jb_p_daily:.4f}")
print(f"Dollar Bar Returns Jarque-Bera: stat={jb_stat_dollar:.2f}, p={jb_p_dollar:.4f}")


## 7. Save Outputs


In [ ]:
pd.DataFrame(index=cusum_events).to_parquet(f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet')
dollar_bars.to_parquet(f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet')
print(f"Saved {TICKER} data to {DATA_CLEAN}")
